# Diagnostico del historico real AI4I

Este notebook documenta decisiones; no contiene datos ni outputs versionados. El pipeline lo ejecuta despues de ingestar el historico persistente y antes de limpiar. `limpiar.py` aplica las mismas funciones sin abrir este notebook.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from kestra.calidad.limpiar import diagnose, clean, NUMERIC
data=Path(os.environ.get('PSET2_DATA_DIR','data'))
raw=pd.read_csv(data/'raw/historico.csv',dtype=str,keep_default_na=False)
audit=diagnose(raw)
print('Filas historicas:',len(raw),'Versiones:',raw.SOURCE_SHA256.nunique())

## 1. Faltantes

Verificacion: nulos y cadenas vacias por columna. Estrategia: rechazar filas incompletas con motivo; no imputar mediciones ni etiquetas sin evidencia. El RAW historico se conserva.

In [ ]:
display(pd.Series(audit['missing_by_column'],name='faltantes').to_frame())

## 2. Duplicados

Verificacion: misma version de fuente y UDI. Estrategia: conservar el menor SOURCE_ROW y auditar las filas descartadas; no deduplicar versiones diferentes ni confundir retries con nuevas observaciones.

In [ ]:
print('Claves de version+UDI repetidas:',audit['duplicate_version_udi'])

## 3. Tipos

Verificacion: casts numericos fallidos. Estrategia: conversion explicita, UDI/rpm/desgaste enteros y flags binarios. Una conversion invalida se rechaza, no se convierte a cero.

In [ ]:
display(pd.Series(audit['failed_numeric_casts'],name='conversiones_fallidas').to_frame())

## 4. Formatos mezclados

Verificacion: comas decimales y espacios. Estrategia: quitar espacios, normalizar coma a punto y tipo de producto a mayusculas. No se interpreta una coma como separador de miles; el contrato de la fuente usa unidades y valores pequenos.

In [ ]:
display(pd.Series(audit['comma_decimal_by_column'],name='coma_decimal').to_frame())

## 5. Validez y consistencia

Exigir tipos L/M/H coherentes con Product ID, Kelvin y rpm positivos, torque/desgaste no negativos y flags 0/1. Conservar las discrepancias target/subtipos: no hay referencia independiente para corregirlas. Los rangos observados no son limites universales.

In [ ]:
good,bad=clean(raw)
print('Validas:',len(good),'Rechazadas:',len(bad))
display(bad.REJECTION_REASON.value_counts().rename('filas').to_frame())
print('Discrepancias conservadas:',int(good.FAILURE_FLAG_MISMATCH.sum()))
display(good[NUMERIC].agg(['min','max']).T)
assert len(raw)==len(good)+len(bad)
assert good.READING_KEY.is_unique

## 6. Archivo de salida y etapa siguiente

`python kestra/calidad/limpiar.py` produce `data/processed/ai4i_clean.csv` y rechazados auditables. Warehouse y Spark consumen exactamente ese archivo; dbt apunta al mismo DuckDB. El notebook no tiene que abrirse para ejecutar la limpieza. El pipeline compara conteos y SHA256 del archivo entre las etapas.